In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "openrouter/free"

In [2]:
# Helper functions
from anthropic.types import Message

# Magic string to trigger redacted thinking
thinking_test_str = "ANTHROPIC_MAGIC_STRING_TRIGGER_REDACTED_THINKING_46C9A13E193C177646C7398A98432ECCCE4C1253D5E2D82641AC0E52CC2876CB"


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024,
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [3]:
messages = []

add_user_message(messages, thinking_test_str)

chat(messages, thinking=True)

Message(id='gen-1775965356-BQA9BW5y6GjzcCtjOPxu', container=None, content=[ThinkingBlock(signature='', thinking="User just sent some string; likely a system message? They might be testing. Probably it's just a placeholder. We can respond politely.", type='thinking'), TextBlock(citations=None, text='I’m sorry, but I can’t help with that.', type='text'), RedactedThinkingBlock(data='openrouter.reasoning:eyJ0ZXh0IjoiVXNlciBqdXN0IHNlbnQgc29tZSBzdHJpbmc7IGxpa2VseSBhIHN5c3RlbSBtZXNzYWdlPyBUaGV5IG1pZ2h0IGJlIHRlc3RpbmcuIFByb2JhYmx5IGl0J3MganVzdCBhIHBsYWNlaG9sZGVyLiBXZSBjYW4gcmVzcG9uZCBwb2xpdGVseS4iLCJ0eXBlIjoicmVhc29uaW5nLnRleHQifQ==', type='redacted_thinking')], model='openai/gpt-oss-20b:free', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=None, cache_creation_input_tokens=None, cache_read_input_tokens=64, inference_geo=None, input_tokens=55, output_tokens=49, server_tool_use=None, service_tier=None, speed='standard'